In [1]:
from pandas_gbq import read_gbq
from pandas_gbq.auth import get_credentials
#import pandas_gbq
import pandas as pd
from pathlib import Path
from datetime import datetime
import time
from io import BytesIO
from google.cloud import storage, bigquery
import numpy as np
import os
import warnings
from google.api_core.exceptions import NotFound, GoogleAPICallError

In [22]:
PROJECT_ID = "rs-nprd-dlk-agspc-roy-5b05"
BUCKET_NAME = "rs-nprd-dlk-ue4-gcs-ryl-sftp_generics"
FOLDER_PATH= "data_entries/TRAMA BS/1. Desgravamen_TC_BN/2025/" 
LOCAL_PATH= "C:/TRAMAS BANCA SEGUROS/EFECTIVA DESGRAVAMEN/2023" 
DATASET_ID = "produccion"
TABLE_ID= "TRAMAS_BS_DESGRAVAMEN_EFECTIVA"
TABLE_CONTROL_ID= "CONTROL_tramas_EF"

In [13]:
# Autenticarse en BIGQUERY
credentials, _ = get_credentials()

bigquery_client = bigquery.Client(
    project=PROJECT_ID,
    credentials=credentials
)

In [23]:

letra = np.array(list("ABCDEFGHIJKLMNOPQRSTUVWXYZ{0123456789"))
valor = np.array(list("1234567891234567890000000000123456789"))
diccionario_reemplazo = dict(zip(letra, valor))

espacios_col = [(0, 3), (3, 7), (7, 23), (48, 49), (43, 45), (45, 48), (368, 376), (376, 384), (384, 392), (346, 347), (639, 641), (396,411),(411, 426)]

column_names = ["Codigo Producto", "Codgo subproducto", "Certificado canal", "Tipo movimiento", "Tipo_registro", "Moneda", 
                "Fecha Afiliacion", "Fecha inicio", "Fecha fin", "Periodo Pago", "Plan seguro", "Suma Asegurada", "Prima Bruta"]

# Convertir cada línea en un dict según los cortes
def Convertir_linea(linea):
    row= {name: linea[start:end].strip() for (start, end), name in zip(espacios_col, column_names)}
    row["Trama Original"] = linea.rstrip("\n")
    return row


def convertir_prima(valor_str):
    try:
        # Debe ser al menos 2 caracteres (números + letra)
        if not isinstance(valor_str, str) or len(valor_str) < 2:
            return np.nan

        parte_numerica = valor_str[:-1]
        letra_final = valor_str[-1]
        # Buscar equivalencia
        digito = diccionario_reemplazo.get(letra_final)
        if digito is None:
            return np.nan  # letra desconocida → nulo
        # Construir número completo
        numero_str = parte_numerica + digito
        # Intentar convertir a Decimal
        return float(numero_str) / 100

    except Exception:
        return np.nan  # En caso de cualquier error, devolver nulo


# Convertir fechas string a tipo datetime
def Convertir_fecha(columna):
    col = columna.astype(str) # Convertir a string
    col = col.str.strip()  # Eliminar espacios
    # Quitar decimales tipo '19880329.0'
    col = col.str.replace(r"\.0$", "", regex=True)
    col = col.str[:8] # Mantener solo los 8 primeros caracteres
    # Reemplazar valores inválidos (que no tengan 8 dígitos) por NaN
    col = col.where(col.str.match(r"^\d{8}$"), np.nan)
    return pd.to_datetime(col, format="%Y%m%d", errors="coerce")  # Convertir a fecha


def buscar_archivo(filename_txt: str) -> bool:
    # Verificar si existe la tabla de control
    table_control_ref = bigquery_client.dataset(DATASET_ID).table(TABLE_CONTROL_ID)
    if not tabla_existe(table_control_ref):
        return False
    else:
        # Buscar archivo en la tabla de control
        query = f"""
            SELECT COUNT(*) AS count
            FROM `{table_control_ref}`
            WHERE ARCHIVO_TXT = @archivo_txt
        """
        job_config = bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("archivo_txt", "STRING", filename_txt)]
        )

        df = bigquery_client.query(query, job_config=job_config).to_dataframe()
        return df["count"].iloc[0] > 0


#### FUNCION PARA COMPROBAR SI LA TABLA ESTA CREADA
def tabla_existe(table_ref):
    try:
      tabla = bigquery_client.get_table(table_ref)
      return True
    except NotFound:
        # La tabla no existe
        return False
    except GoogleAPICallError as e:
        # Otros errores de BigQuery (permisos, conexión, etc.)
        print(f"⚠️ Error al consultar BigQuery: {e}")
        raise

#### FUNCION PARA GUARDAR UN DATASET EN UNA TABLA DE BIGQUERY
def Guardar_en_BigQuery(data, dataset_id, table_id):
    table_ref = bigquery_client.dataset(dataset_id).table(table_id)
    job_config = bigquery.LoadJobConfig()
    if tabla_existe(table_ref):
        # Abre la tabla para agregar registros
        job_config.write_disposition = bigquery.WriteDisposition.WRITE_APPEND
    else:
        tabla_tramas = bigquery.Table(table_ref)
        tabla_tramas = bigquery_client.create_table(tabla_tramas)
        job_config.write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE
        job_config.autodetect = False
        print(f'   ℹ️ ... Se ha creado la tabla: {table_id} en el dataset: {dataset_id} ...')
    job = bigquery_client.load_table_from_dataframe(data, table_ref, job_config=job_config)
    job.result()
    return

#### PROCESAR ARCHIVOS
carpeta= Path(LOCAL_PATH)
for archivo in carpeta.iterdir():
    if archivo.is_file() and archivo.suffix.lower() == ".txt":
        file_name= archivo.name
        if not buscar_archivo(file_name):
            print(f"⏳ ... PROCESANDO ARCHIVO: {file_name} ... ⏳")
            with open(archivo, "r", encoding="latin-1") as file:
                lista_lineas = file.readlines()
                df_tramas = pd.DataFrame([Convertir_linea(linea) for linea in lista_lineas])
                df_tramas.columns = (df_tramas.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True)  # reemplazar todo lo que no sea letra/número/por _
                                        )
                df_tramas["PRIMA_BRUTA"] = df_tramas["PRIMA_BRUTA"].apply(convertir_prima)
                df_tramas["SUMA_ASEGURADA"] = df_tramas["SUMA_ASEGURADA"].apply(convertir_prima)
                df_tramas["FECHA_AFILIACION"] = Convertir_fecha(df_tramas["FECHA_AFILIACION"]).dt.date
                df_tramas["FECHA_INICIO"] = Convertir_fecha(df_tramas["FECHA_INICIO"]).dt.date
                df_tramas["FECHA_FIN"] = Convertir_fecha(df_tramas["FECHA_FIN"]).dt.date
                df_tramas["NOMBRE_ARCHIVO"] = file_name

                fecha_carga = datetime.now()
                df_control = pd.DataFrame([{"ARCHIVO_TXT": file_name, "FECHA_CARGA": fecha_carga}])
                Guardar_en_BigQuery(df_tramas, DATASET_ID, TABLE_ID)
                Guardar_en_BigQuery(df_control, DATASET_ID, TABLE_CONTROL_ID)
                print(f"✅ ARCHIVO {file_name} CARGADO ...")
        else:
            print(f"⚠️ EL ARCHIVO {file_name} YA FUE CARGADO ANTERIORMENTE ...")



⏳ ... PROCESANDO ARCHIVO: ALTAS_DESGRAVAMEN_ABRIL_2023.TXT ... ⏳
✅ ARCHIVO ALTAS_DESGRAVAMEN_ABRIL_2023.TXT CARGADO ...
⏳ ... PROCESANDO ARCHIVO: ALTAS_DESGRAVAMEN_MARZO_2023.TXT ... ⏳
✅ ARCHIVO ALTAS_DESGRAVAMEN_MARZO_2023.TXT CARGADO ...
⏳ ... PROCESANDO ARCHIVO: ALTAS_DESGRAVAMEN_MAYO_2023.TXT ... ⏳
✅ ARCHIVO ALTAS_DESGRAVAMEN_MAYO_2023.TXT CARGADO ...
⏳ ... PROCESANDO ARCHIVO: ALTA_DESGRAVAMEN_JULIO_2023.txt ... ⏳
✅ ARCHIVO ALTA_DESGRAVAMEN_JULIO_2023.txt CARGADO ...
⏳ ... PROCESANDO ARCHIVO: ALTA_DESGRAVAMEN_JUNIO_2023.txt ... ⏳
✅ ARCHIVO ALTA_DESGRAVAMEN_JUNIO_2023.txt CARGADO ...
⏳ ... PROCESANDO ARCHIVO: Alta_Desgravamen_Multiproductos _Noviembre_2023.txt ... ⏳
✅ ARCHIVO Alta_Desgravamen_Multiproductos _Noviembre_2023.txt CARGADO ...
⏳ ... PROCESANDO ARCHIVO: TRAMA_ALTA_DESGRAVAMEN_AGOSTO_2023.txt ... ⏳
✅ ARCHIVO TRAMA_ALTA_DESGRAVAMEN_AGOSTO_2023.txt CARGADO ...
⏳ ... PROCESANDO ARCHIVO: TRAMA_ALTA_DESGRAVAMEN_DICIEMBRE_2022.txt ... ⏳
✅ ARCHIVO TRAMA_ALTA_DESGRAVAMEN_DICIEMBRE

## TEST

In [5]:
trama= "C:/Datos/ALTA_DESGRAVAMEN_ABRIL_2026.txt"

In [6]:
with open(trama, "r", encoding="latin-1") as file:
    lista_lineas = file.readlines()[1:]

In [7]:
len(lista_lineas)

58719

In [8]:
df_tramas = pd.DataFrame([Convertir_linea(linea) for linea in lista_lineas])
df_tramas.columns = (df_tramas.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True)  # reemplazar todo lo que no sea letra/número/por _
                        )

In [9]:
df_tramas.head(3)

,CODIGO_PRODUCTO,CERTIFICADO_CANAL,TIPO_MOVIMIENTO,TIPO_REGISTRO,MONEDA,FECHA_AFILIACION,FECHA_INICIO,FECHA_FIN,PERIODO_PAGO,PLAN_SEGURO,SUMA_ASEGURADA,PRIMA_BRUTA,TRAMA_ORIGINAL
0,980,00020000000002511731,1,01,PEN,20260401,20260401,20280403,M,01,000000000573025,000000000002292,980000200000000025117310000000000000251173101P...
1,980,00020000000002514000,1,01,PEN,20260401,20260401,20270405,M,01,000000000100000,000000000000400,980000200000000025140000000000000000251400001P...
2,980,00020000000002514001,1,01,PEN,20260401,20260401,20260815,M,01,000000000080000,000000000000320,980000200000000025140010000000000000251400101P...


In [10]:
df_tramas["PRIMA_BRUTA"] = df_tramas["PRIMA_BRUTA"].apply(convertir_prima)
df_tramas["SUMA_ASEGURADA"] = df_tramas["SUMA_ASEGURADA"].apply(convertir_prima)
df_tramas["FECHA_INICIO"] = Convertir_fecha(df_tramas["FECHA_INICIO"]).dt.date
df_tramas["FECHA_FIN"] = Convertir_fecha(df_tramas["FECHA_FIN"]).dt.date

In [11]:
df_tramas.head(3)

,CODIGO_PRODUCTO,CERTIFICADO_CANAL,TIPO_MOVIMIENTO,TIPO_REGISTRO,MONEDA,FECHA_AFILIACION,FECHA_INICIO,FECHA_FIN,PERIODO_PAGO,PLAN_SEGURO,SUMA_ASEGURADA,PRIMA_BRUTA,TRAMA_ORIGINAL
0,980,00020000000002511731,1,01,PEN,20260401,2026-04-01,2028-04-03,M,01,5730.25,22.92,980000200000000025117310000000000000251173101P...
1,980,00020000000002514000,1,01,PEN,20260401,2026-04-01,2027-04-05,M,01,1000.00,4.00,980000200000000025140000000000000000251400001P...
2,980,00020000000002514001,1,01,PEN,20260401,2026-04-01,2026-08-15,M,01,800.00,3.20,980000200000000025140010000000000000251400101P...


In [36]:
Guardar_en_BigQuery(df_tramas, DATASET_ID, TABLE_ID)

   ℹ️ ... Se ha creado la tabla: TRAMAS_BS_DESGRAVAMEN_TC_BN-2 en el dataset: produccion ...


In [14]:
df_tramas.info()

<class 'pandas.DataFrame'>
RangeIndex: 58719 entries, 0 to 58718
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CODIGO_PRODUCTO    58719 non-null  str    
 1   CERTIFICADO_CANAL  58719 non-null  str    
 2   TIPO_MOVIMIENTO    58719 non-null  str    
 3   TIPO_REGISTRO      58719 non-null  str    
 4   MONEDA             58719 non-null  str    
 5   FECHA_AFILIACION   58719 non-null  str    
 6   FECHA_INICIO       58719 non-null  object 
 7   FECHA_FIN          58719 non-null  object 
 8   PERIODO_PAGO       58719 non-null  str    
 9   PLAN_SEGURO        58719 non-null  str    
 10  SUMA_ASEGURADA     58719 non-null  float64
 11  PRIMA_BRUTA        58719 non-null  float64
 12  TRAMA_ORIGINAL     58719 non-null  str    
dtypes: float64(2), object(2), str(9)
memory usage: 45.0+ MB
